In [1]:
from datasets import load_dataset
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tinystories_dataset.shape

(2119719, 1)

In [3]:
gsm8k_dataset.shape


(7473, 2)

In [4]:
tinystories_dataset[0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

In [5]:
gsm8k_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [6]:
def get_training_corpus(tinystories_dataset, gsm8k_dataset):
    for i, row in enumerate(tinystories_dataset):
        yield row['text']
    for row in gsm8k_dataset:
        question = row['question']
        raw_answer = row['answer']

        parts = raw_answer.split('####')
        reasoning = parts[0].strip()
        final_answer = parts[1].strip() if len(parts) > 1 else ""

        formatted_text = (
            f"[INST] {question} [/INST]\n"
            f"[THINK]\n{reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        yield formatted_text
        


In [45]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Sequence, Whitespace, Split
from tokenizers.processors import TemplateProcessing
from tokenizers import decoders

def train_hf_tokenizer(output_dir: str = "./trm_tokenizer"):
    print("Initializing BPE Tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]", continuing_subword_prefix="##") )

    # The Math Hack: Split by whitespace, then isolate digits
    tokenizer.pre_tokenizer = Sequence([
        Whitespace(),
        Split(Regex(r"\d"), behavior="isolated") 
    ])

    # Structural Tokens for G-TRM
    special_tokens = [
        "[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]", 
        "[INST]", "[/INST]", "[THINK]", "[/THINK]", "[ANSWER]", "[/ANSWER]"
    ]

    # 4096 Vocab Size to keep embedding layer ~1M parameters
    trainer = BpeTrainer(
        vocab_size=4096, 
        special_tokens=special_tokens,
        initial_alphabet=list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*()_+-=[]{}|;':\",./<>?`~ \n"),
        show_progress=True,
        continuing_subword_prefix="##"
    )

    tokenizer.decoder = decoders.WordPiece(prefix="##")

    print("Training tokenizer from Hugging Face iterator...")
    # Train using the generator!
    tokenizer.train_from_iterator(get_training_corpus(tinystories_dataset, gsm8k_dataset), trainer=trainer)

    # Post-Processing: Auto-wrap sequences in BOS and EOS
    tokenizer.post_processor = TemplateProcessing(
        single="[BOS] $A [EOS]",
        pair="[BOS] $A [EOS] $B:1 [EOS]:1",
        special_tokens=[
            ("[BOS]", tokenizer.token_to_id("[BOS]")),
            ("[EOS]", tokenizer.token_to_id("[EOS]")),
        ],
    )

    # Save to disk
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    save_path = os.path.join(output_dir, "tokenizer_mask_1.json")
    tokenizer.save(save_path)
    print(f"\nSuccess! Custom Micro-BPE tokenizer saved to {save_path}")

    return tokenizer

In [46]:
my_tokenizer = train_hf_tokenizer()

Initializing BPE Tokenizer...
Training tokenizer from Hugging Face iterator...

Success! Custom Micro-BPE tokenizer saved to ./trm_tokenizer\tokenizer_mask_1.json


Preparing Dataset

In [11]:
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
import re

# ==========================================
# 1. Configuration & Tokenizer Setup
# ==========================================
MAX_SEQ_LEN = 512 
BATCH_SIZE = 16

print("Loading Tokenizer...")
tokenizer = Tokenizer.from_file("C:/Users/MSI-1/Desktop/Adnan's Major/TinyRecursiveModels/trm_tokenizer/tokenizer_mask_1.json")

# Extract special token IDs (assuming your tokenizer has these defined)
MASK_TOKEN_ID = tokenizer.token_to_id("[MASK]")
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
IGNORE_INDEX = -100 # PyTorch standard for ignoring loss

# ==========================================
# 2. Formatting & Tokenization Logic
# ==========================================
def process_gsm8k_nar(examples):
    """Processes GSM8K directly into padded/masked NAR tensors."""
    inputs, labels = [], []
    
    for question, answer in zip(examples["question"], examples["answer"]):
        # 1. Split and clean
        parts = answer.split("####")
        raw_reasoning = parts[0].strip()
        clean_reasoning = re.sub(r'<<.*?>>', '', raw_reasoning)
        final_answer = parts[1].strip() if len(parts) > 1 else ""
        
        # 2. Define the Prompt vs the Full Target
        prompt_text = f"[INST] {question} [/INST]\n"
        full_text = (
            f"{prompt_text}"
            f"[THINK]\n{clean_reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        
        # 3. Tokenize separately
        prompt_ids = tokenizer.encode(prompt_text).ids[:MAX_SEQ_LEN]
        full_ids = tokenizer.encode(full_text).ids[:MAX_SEQ_LEN]
        
        # 4. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 5. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)
        
    return {"input_ids": inputs, "labels": labels}

def process_tinystories_nar(examples):
    """Processes TinyStories. We extract the first few words as a prompt."""
    inputs, labels = [], []
    
    for text in examples["text"]:
        # 1. Define the full target
        full_ids = tokenizer.encode(text).ids[:MAX_SEQ_LEN]
        
        # 2. Create a prompt (e.g., the first 8 tokens to kickstart generation)
        # If you want unconditional generation, just use an empty list or [BOS] token here.
        prompt_ids = full_ids[:8] 
        
        # 3. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 4. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)

    return {"input_ids": inputs, "labels": labels}

# ==========================================
# 3. Execution Pipeline
# ==========================================
print("Loading Datasets...")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train[:100000]")

print("Processing into NAR Tensors... (This merges formatting and tokenizing)")
# Map the new NAR processing functions directly
gsm8k_tokenized = gsm8k_dataset.map(
    process_gsm8k_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=gsm8k_dataset.column_names # Strip all old columns
)

tinystories_tokenized = tinystories_dataset.map(
    process_tinystories_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=tinystories_dataset.column_names # Strip all old columns
)

print("Formatting for PyTorch...")
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])
tinystories_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

# Now, a single batch gives you perfectly sized [B, 512] tensors ready for TRM!

Loading Tokenizer...
Loading Datasets...
Processing into NAR Tensors... (This merges formatting and tokenizing)


Map: 100%|██████████| 100000/100000 [00:36<00:00, 2770.95 examples/s]

Formatting for PyTorch...


In [12]:
tokenizer.decode(gsm8k_tokenized[0]['input_ids'].tolist())


'Natalia sold clips to 4 8 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [13]:
fix = torch.where(gsm8k_tokenized[0]['labels'] != -100,gsm8k_tokenized[0]['labels'],0)
tokenizer.decode(fix.tolist())

'Natalia sold clips to 4 8 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Natalia sold 4 8 / 2 = 2 4 clips in May. Natalia sold 4 8 + 2 4 = 7 2 clips altogether in April and May. 7 2'

In [14]:
gsm8k_tokenized[0]['input_ids'].tolist()

[2,
 5,
 58,
 257,
 330,
 679,
 2044,
 436,
 2992,
 244,
 32,
 36,
 290,
 274,
 428,
 280,
 45,
 163,
 158,
 255,
 24,
 242,
 528,
 294,
 2044,
 1993,
 358,
 643,
 436,
 2992,
 280,
 4066,
 26,
 1147,
 643,
 436,
 2992,
 405,
 58,
 257,
 330,
 679,
 2034,
 447,
 625,
 496,
 280,
 45,
 163,
 158,
 255,
 242,
 4066,
 43,
 6,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,

In [15]:
from torch.utils.data import DataLoader

gsm8k_loader = DataLoader(gsm8k_tokenized, batch_size=BATCH_SIZE, shuffle=True)
tinystories_loader = DataLoader(tinystories_tokenized, batch_size=BATCH_SIZE, shuffle=True)

print("\n--- Pipeline Complete! ---")
print(f"GSM8K Batches available: {len(gsm8k_loader)}")
print(f"TinyStories Batches available: {len(tinystories_loader)}")

# Let's verify a batch!
sample_batch = next(iter(gsm8k_loader))
print(f"\nShape of input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape of labels: {sample_batch['labels'].shape}")
print(f"First 10 input_ids: {sample_batch['input_ids'][0][:10]}")
print(f"Last 10 labels (Notice the -100 padding!): {sample_batch['labels'][0][-10:]}")


--- Pipeline Complete! ---
GSM8K Batches available: 468
TinyStories Batches available: 6250

Shape of input_ids: torch.Size([16, 512])
Shape of labels: torch.Size([16, 512])
First 10 input_ids: tensor([   2,    5, 1071,  393, 1015, 1796,  358,  643, 1553,  270])
Last 10 labels (Notice the -100 padding!): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


In [16]:
config_dict = {
    # 1. Sequence & Vocabulary (The Generative Shift)
    "batch_size": 16,               
    "seq_len": 512,                 # Maximum context window for text
    "vocab_size": 4096,             # Must match your Micro-BPE tokenizer!
    
    # 2. Transformer Core (~7M Params)
    "hidden_size": 512,             
    "expansion": 4.0,               
    "num_heads": 8,                 
    "pos_encodings": "rope",        # CRITICAL: Replaces 'none' from Sudoku
    
    # 3. Recursion & Loops
    "L_layers": 2,                  # Physical layers
    "H_layers": 1,                  # (Ignored)
    "L_cycles": 6,                  # n
    "H_cycles": 3,                 # T
    
    # 4. Adaptive Computation Time (ACT)
    "halt_max_steps": 16,           # Nsupp
    "halt_exploration_prob": 0.1,   
    
    # 5. Stability & Normalization
    "rms_norm_eps": 1e-5,
    "rope_theta": 10000.0,
    "forward_dtype": "bfloat16",    
    
    # 6. Puzzle Disablement (Turning off Sudoku features)
    "mlp_t": False,                 # CRITICAL: Must be False to use Attention
    "puzzle_emb_ndim": 0,           # Turn off puzzle embeddings
    "num_puzzle_identifiers": 1,    # Safe dummy value
    "puzzle_emb_len": 0,
    "no_ACT_continue": True, 
    "causal": False    
}

In [17]:
@torch.no_grad()
def generate(self, input_ids: torch.Tensor, max_new_tokens: int, iterations: int = 30, **kwargs): # Increased iterations to 30
    self.eval()
    b_size = input_ids.shape[0]
        
    mask_token_id = tokenizer.token_to_id("[MASK]")
    pad_token_id = tokenizer.token_to_id("[PAD]")
    eos_token_id = tokenizer.token_to_id("[EOS]")
        
    current_masks = torch.full((b_size, max_new_tokens), mask_token_id, dtype=torch.long, device=input_ids.device)
    current_sequence = torch.cat([input_ids, current_masks], dim=1)
        
    for step in range(iterations):
        batch = {
            "inputs": current_sequence,
            "puzzle_identifiers": torch.zeros((b_size, self.config.num_puzzle_identifiers), device=input_ids.device, dtype=torch.int32)
            }
        carry = self.initial_carry(batch)
            
        for _ in range(self.config.halt_max_steps):
            carry, outputs = self.forward(carry, batch)
            if carry.halted.all():
                break
            
        logits = outputs["logits"]
        logits[:, :, mask_token_id] = -float('inf')
            
        probs = torch.softmax(logits, dim=-1)
        confidence, predicted_tokens = torch.max(probs, dim=-1)
            
        gen_start_idx = input_ids.shape[1]
        generated_preds = predicted_tokens[:, gen_start_idx:]
        generated_conf = confidence[:, gen_start_idx:]
            
            # --- THE NAR REPETITION PENALTY ---
            # Artificially shatter the confidence of repeating tokens so they get re-masked
        for b in range(b_size):
            for seq_idx in range(1, generated_preds.shape[1]):
                if generated_preds[b, seq_idx] == generated_preds[b, seq_idx-1]:
                    # If a token repeats, drop its confidence significantly
                    generated_conf[b, seq_idx] *= 0.2 
            
            # The [EOS] Enforcer
        for b in range(b_size):
            eos_positions = (generated_preds[b] == eos_token_id).nonzero(as_tuple=True)[0]
            if len(eos_positions) > 0:
                first_eos = eos_positions[0]
                generated_preds[b, first_eos+1:] = pad_token_id
                generated_conf[b, first_eos+1:] = 1.0 
            
        ratio_to_mask = 1.0 - ((step + 1) / iterations)
        num_to_mask = int(max_new_tokens * ratio_to_mask)
            
        if num_to_mask > 0:
            lowest_conf_indices = torch.topk(-generated_conf, k=num_to_mask, dim=-1).indices
            current_sequence[:, gen_start_idx:] = generated_preds
            current_sequence[:, gen_start_idx:].scatter_(1, lowest_conf_indices, mask_token_id)
        else:
            current_sequence[:, gen_start_idx:] = generated_preds

    final_generation = current_sequence[:, input_ids.shape[1]:]
    return final_generation

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
import math


def cosine_schedule_with_warmup_lr_lambda(
    current_step: int,
    *,
    base_lr: float,
    num_warmup_steps: int,
    num_training_steps: int,
    min_ratio: float = 0.0,
    num_cycles: float = 0.5,
):
    """Same schedule as pretrain.py: linear warmup then cosine decay."""
    current_step = min(max(int(current_step), 0), int(num_training_steps))
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


from upload import TinyRecursiveReasoningModel_ACTV1
from ema import EMAHelper

# Assume your model and loaders are imported/defined above
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
# gsm8k_loader, tinystories_loader = ... 

# ==========================================
# 1. Initialization & VRAM Optimization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Initialize Model (Make sure config has causal=True and vocab_size=4096)
model = TinyRecursiveReasoningModel_ACTV1(config_dict).to(device)

# Optimizer LR is set each update step (see train_epoch + lr_schedule), matching pretrain.py style.
optimizer = AdamW(model.parameters(), lr=0.0, weight_decay=0.1)

# Mixed Precision Scaler (Crucial for 4GB/8GB GPUs)
scaler = torch.amp.GradScaler('cuda')

# Gradient Accumulation: Simulates a larger batch size without crashing VRAM
accumulation_steps = 4

# LR schedule hyperparams (pretrain.py: lr, lr_warmup_steps, lr_min_ratio)
base_lr_phase1 = 5e-4
base_lr_phase2 = 1e-4
lr_warmup_steps = 2000
lr_min_ratio = 0.0
lr_schedule = {
    "step": 0,
    "total_steps": 1,
    "base_lr": base_lr_phase1,
    "warmup_steps": 1,
    "min_ratio": lr_min_ratio,
}

# EMA (same flags / helper as pretrain.py PretrainConfig.ema, ema_rate)
ema = False
ema_rate = 0.999
ema_helper = None
if ema:
    print("Setup EMA")
    ema_helper = EMAHelper(mu=ema_rate)
    ema_helper.register(model)


def save_training_checkpoint(path, model, ema_helper):
    """Save EMA weights when EMA is enabled (mirrors pretrain checkpointing with ema_copy)."""
    if ema_helper is not None:
        ema_model = ema_helper.ema_copy(model)
        try:
            torch.save(ema_model.state_dict(), path)
        finally:
            del ema_model
    else:
        torch.save(model.state_dict(), path)


Training on device: cuda


In [ ]:
def train_epoch(model, dataloader, optimizer, scaler, epoch, phase_name, ema_helper=None, lr_schedule=None):
    model.train()
    total_loss = 0
    
    mask_token_id = tokenizer.token_to_id("[MASK]")
    pad_token_id = tokenizer.token_to_id("[PAD]")
    
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        
        # --- THE PADDING LEAK FIX ---
        # Clone labels so we can modify them without breaking the dataset
        labels = batch["labels"].clone().to(device)
        
        # Find where the sequence is padded. 
        # Change the label from -100 to pad_token_id so the model is 
        # actively rewarded for predicting [PAD] in empty space.
        is_pad = (input_ids == pad_token_id)
        labels[is_pad] = pad_token_id
        
        # Dynamic NAR Masking Logic
        mask_ratio = torch.empty(1).uniform_(0.1, 1.0).item()
        prob_matrix = torch.full(labels.shape, mask_ratio, device=device)
        
        # ONLY protect the prompt (which is STILL -100) from masking.
        # Now, [PAD] tokens will be masked! The model will see 100% masks
        # and learn to generate [EOS] and [PAD] to establish its own bounds.
        prob_matrix.masked_fill_(labels == -100, 0.0) 
        masked_indices = torch.bernoulli(prob_matrix).bool()
        
        masked_inputs = input_ids.clone()
        masked_inputs[masked_indices] = mask_token_id
        
        trm_batch = {
            "inputs": masked_inputs,
            "puzzle_identifiers": torch.zeros((input_ids.shape[0], model.config.num_puzzle_identifiers), dtype=torch.int32, device=device)
        }
        carry = model.initial_carry(trm_batch)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            _, outputs = model(carry, trm_batch)
            logits = outputs["logits"]
            
            # Loss now calculates penalty for getting [PAD] wrong
            loss = F.cross_entropy(
                logits.view(-1, model.config.vocab_size), 
                labels.view(-1)
            )
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if lr_schedule is not None:
                lr_schedule["step"] += 1
                s = lr_schedule["step"]
                lr_this = cosine_schedule_with_warmup_lr_lambda(
                    s,
                    base_lr=lr_schedule["base_lr"],
                    num_warmup_steps=lr_schedule["warmup_steps"],
                    num_training_steps=lr_schedule["total_steps"],
                    min_ratio=lr_schedule["min_ratio"],
                )
                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr_this
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} [{phase_name}] | Batch {batch_idx}/{len(dataloader)} | Mask: {mask_ratio*100:.0f}% | Loss: {loss.item() * accumulation_steps:.4f}")

        if ema_helper is not None:
            ema_helper.update(model)

    return total_loss / len(dataloader)

In [ ]:

num_tinystories_epochs = 2
num_gsm8k_epochs = 5


def _optimizer_steps_per_phase(loader, accum, epochs):
    """Optimizer steps per phase (matches (batch_idx+1) % accum == 0 counting)."""
    return max(1, (len(loader) // accum) * epochs)


print("\n=== PHASE 1: Syntax (TinyStories) ===")
lr_schedule["step"] = 0
lr_schedule["base_lr"] = base_lr_phase1
lr_schedule["total_steps"] = _optimizer_steps_per_phase(tinystories_loader, accumulation_steps, num_tinystories_epochs)
lr_schedule["warmup_steps"] = min(lr_warmup_steps, lr_schedule["total_steps"])
lr_schedule["min_ratio"] = lr_min_ratio

for epoch in range(1, num_tinystories_epochs + 1):
    avg_loss = train_epoch(model, tinystories_loader, optimizer, scaler, epoch, "TinyStories", ema_helper, lr_schedule)
    print(f"--> Phase 1, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

    # NEW: Save Phase 1 weights (EMA copy when ema=True, same as pretrain checkpointing)
    save_training_checkpoint(f"g_trm_tinystories_epoch_{epoch}.pt", model, ema_helper)

print("\n=== PHASE 2: Reasoning (GSM8K) ===")
# New phase: fresh warmup + cosine to base_lr_phase2 (same pattern as pretrain schedule)
lr_schedule["step"] = 0
lr_schedule["base_lr"] = base_lr_phase2
lr_schedule["total_steps"] = _optimizer_steps_per_phase(gsm8k_loader, accumulation_steps, num_gsm8k_epochs)
lr_schedule["warmup_steps"] = min(lr_warmup_steps, lr_schedule["total_steps"])
lr_schedule["min_ratio"] = lr_min_ratio

for epoch in range(1, num_gsm8k_epochs + 1):
    avg_loss = train_epoch(model, gsm8k_loader, optimizer, scaler, epoch, "GSM8K", ema_helper, lr_schedule)
    print(f"--> Phase 2, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")
    
    # Save checkpoint
    save_training_checkpoint(f"g_trm_gsm8k_epoch_{epoch}.pt", model, ema_helper)

print("Training Complete!")


=== PHASE 1: Syntax (TinyStories) ===
Epoch 1 [TinyStories] | Batch 0/6250 | Mask: 97% | Loss: 9.0039
Epoch 1 [TinyStories] | Batch 50/6250 | Mask: 93% | Loss: 6.1267
Epoch 1 [TinyStories] | Batch 100/6250 | Mask: 34% | Loss: 6.0736


In [ ]:
import torch
import re
from datasets import load_dataset
from tokenizers import Tokenizer
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1

# ==========================================
# 1. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing Evaluation Pipeline...")

# Load your custom tokenizer
tokenizer = Tokenizer.from_file("./trm_tokenizer/tokenizer_mask.json")

# Load model (Use the exact config_dict from your training script)
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_gsm8k_epoch_5.pt", map_location=device))
model.to(device)
model.eval()

# Load the GSM8K Test Split (Not the train split!)
test_dataset = load_dataset("gsm8k", "main", split="test")

# ==========================================
# 2. Answer Extraction Logic
# ==========================================
def extract_model_answer(generated_text: str):
    """Uses Regex to find the number inside the [ANSWER] tags."""
    match = re.search(r'\[ANSWER\](.*?)\[/ANSWER\]', generated_text, re.DOTALL)
    if match:
        numbers = re.findall(r'-?\d+\.?\d*', match.group(1))
        return float(numbers[-1]) if numbers else None
    return None

def extract_ground_truth(raw_answer: str):
    """Extracts the final number after #### in the GSM8K dataset."""
    parts = raw_answer.split("####")
    if len(parts) > 1:
        numbers = re.findall(r'-?\d+\.?\d*', parts[1])
        return float(numbers[-1]) if numbers else None
    return None

# ==========================================
# 3. The Evaluation Loop
# ==========================================
total_questions = 0
correct_answers = 0

# Dictionaries to track the "Inference Gap"
difficulty_tracker = {"Easy (1-2 steps)": [0, 0], "Medium (3-4 steps)": [0, 0], "Hard (5+ steps)": [0, 0]}

print(f"Starting evaluation on {len(test_dataset)} test questions...\n")

for i, example in enumerate(test_dataset):
    question = example["question"]
    ground_truth_text = example["answer"]
    
    num_steps = ground_truth_text.split("####")[0].count('\n')
    if num_steps <= 2:
        difficulty_category = "Easy (1-2 steps)"
    elif num_steps <= 4:
        difficulty_category = "Medium (3-4 steps)"
    else:
        difficulty_category = "Hard (5+ steps)"

    prompt = f"[INST] {question} [/INST]\n[THINK]"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)

    # Generate with strict temperature and NO repetition penalty
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=100)
    
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    
    model_ans = extract_model_answer(generated_text)
    truth_ans = extract_ground_truth(ground_truth_text)

    # ==========================================
    # INDENTATION FIXED BELOW
    # ==========================================
    total_questions += 1
    difficulty_tracker[difficulty_category][1] += 1
    
    is_correct = False
    if model_ans is not None and truth_ans is not None:
        if model_ans == truth_ans:
            correct_answers += 1
            difficulty_tracker[difficulty_category][0] += 1
            is_correct = True

    # if i % 10 == 0 or model_ans is None: 
    print(f"\n[{i}/{len(test_dataset)}] Accuracy so far: {(correct_answers/total_questions)*100:.2f}%")
    if not is_correct:
        print(f"  -> Expected: {truth_ans}")
        print(f"  -> Extracted: {model_ans}")
        print(f"  -> RAW MODEL OUTPUT:\n{generated_text}\n")
        print("-" * 40)

# ==========================================
# 4. Final Thesis Metrics Output
# ==========================================
final_accuracy = (correct_answers / total_questions) * 100
print("\n" + "="*50)
print(f"FINAL MODEL ACCURACY: {final_accuracy:.2f}%")
print("="*50)
print("INFERENCE GAP ANALYSIS (For Presentation Graph):")
for category, counts in difficulty_tracker.items():
    correct, total = counts
    if total > 0:
        acc = (correct / total) * 100
        print(f"- {category}: {acc:.2f}% ({correct}/{total})")